In [1]:
!pip install -q kokoro>=0.9.2 soundfile
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

In [2]:
!pip install ultralytics transformers torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.6 MB/s eta 0:00:00


In [3]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 54.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 25.6 MB/s eta 0:00:00


In [1]:
import streamlit as st
from transformers import pipeline, AutoProcessor, AutoModelForImageTextToText
from ultralytics import YOLO
from PIL import Image
import cv2
import numpy as np
import torch
import soundfile as sf
from kokoro import KPipeline
from IPython.display import display, Audio
import soundfile as sf
import os
import base64


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
##################################################################################################################################
##### using the hugging face model : mosesb/best-comic-panel-detection
##################################################################################################################################

def detect_panels(input_image):

    panels = {}
    height, width = input_image.shape[:2]
    image_area = float(width*height)

    model = YOLO('best.pt')
    results = model.predict(input_image)

    proximity_threshold = 50

    # Lets only consider the panels that are reasonable in size
    for result in results:
        boxes = result.boxes
        if boxes is not None:
            for box in boxes:
                index = len(panels)
                # get the panel coordinates
                x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
                # converting the coordinates to integers for further processing
                x, y = int(x1) , int(y1)
                w, h = int(x2-x1) , int(y2-y1)
                area = float(w*h)
                # trying to clear the panels that are of unreasonable size
                if area/image_area > 0.01 and area/image_area < 0.95:
                    if w>10 and h>10:
                        panels[index] = [x,y,w,h]

    # sort them based on the reading order
    PanelList = list(panels.values())

    sortedPanelList = westernReadingOrder(PanelList, proximity_threshold)
    return sortedPanelList

##################################################################################################################################

def westernReadingOrder(PanelList, proximity_threshold = 50):


    print(PanelList)

    # Sort Panel list by top to bottom
    PanelList.sort(key = lambda x : x[1])

    print(f"Panel List after sorting top to bottom {PanelList}")

    # group Panels into rows
    rows = []
    current_row = [PanelList[0]]
    for i in range(1, len(PanelList)):
        # if panels y-top is close to the previous panel y-top, they are in one row
        if abs(PanelList[i][1] - current_row[-1][1]) < proximity_threshold:
            current_row.append(PanelList[i])
        else:
            # sort the completed row horizontally ( left to right)
            current_row.sort(key=lambda x:x[0])
            rows.extend(current_row)
            current_row = [PanelList[i]]

    # sort and add the final row
    current_row.sort(key = lambda x:x[0])
    rows.extend(current_row)

    # sorting contours to read from left to right
    #PanelList.sort(key = lambda x:get_contour_precedence(x, width))

    print(rows)
    return rows

##################################################################################################################################

# comparision function for sorting contours
def get_contour_precedence(contour, cols):
    tolerance_factor = 200
    [x, y, w, h] = contour
    return((y//tolerance_factor)* tolerance_factor)*cols+x

##################################################################################################################################

def cropImages(image, contours, padding = 0):
    print("Cropping Images")
    croppedImageList = []
    i = 0
    for contour in contours:#for contour in contours.xyxy:
        print(contour)
        i = i + 1
        [x, y, w, h] = contour
        croppedImage = image[y-padding : y+h+padding, x-padding:x+w+padding]
        croppedImageList.append(croppedImage)

        cv2.imwrite("Panel"+str(i)+".png", croppedImage)

    return croppedImageList

##################################################################################################################################
##### EXTRACTS TEXT FROM THE PANELS
##################################################################################################################################

def extractText(ImageList , textfilename):

    print("Extracting Text")

    MODEL_PATH = "zai-org/GLM-OCR"

    processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        pretrained_model_name_or_path=MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

    for i in range(len(ImageList)):
        image_file = "Panel" + str(i+1) + ".png"


        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "url": image_file
                    },
                    {
                        "type": "text",
                        "text": "Text Recognition:"
                    }
                ],
            }
        ]



        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        inputs.pop("token_type_ids", None)

        generated_ids = model.generate(**inputs, max_new_tokens=8192)
        output_text = processor.decode(
            generated_ids[0][inputs["input_ids"].shape[1] :],
            skip_special_tokens=True,
        )

        if not os.path.exists(textfilename):
            with open(textfilename, 'w') as f:
                f.write(output_text)
        else:
            with open(textfilename, 'a') as f:
                f.write(output_text)

#############################################################################################################################
##### CONVERTS THE TEXT TO THE AUDIO FILE
##################################################################################################################################

def text_to_speech_cpu(textfilename, output_file="output.wav"):

    audiolist = []

    print("Creating Audio File")

    if not os.path.exists(textfilename):
        print(f"Error: The file '{textfilename}' was not found.")
        return

    # 2. Read text from the file
    with open(textfilename, 'r', encoding='utf-8') as f:
        text_content = f.read()

    text_content = text_content.replace('\n', ' ')

    if not text_content:
        print("Error: The text file is empty.")
        return

    pipeline = KPipeline(lang_code='a')

    generator = pipeline(text_content, voice='af_heart')
    for i, (gs, ps, audio) in enumerate(generator):
        print(i, gs, ps)
        audiolist.append(audio)

    concatenated_data = np.vstack(audiolist)
    display(Audio(data=concatenated_data, rate=24000, autoplay=i==0))
    sf.write(output_file, np.ravel(concatenated_data), 24000)


In [3]:
input = "comicPage1.png"
# convert image into numpy array
input_image = cv2.imread(input)
panels = detect_panels(input_image)
cropped_images = cropImages(input_image, panels)
textfilename = "ComicText.txt"
extractText(cropped_images , textfilename)
text_to_speech_cpu(textfilename)



0: 640x416 6 Comic Panels, 4693.1ms
Speed: 17.5ms preprocess, 4693.1ms inference, 66.5ms postprocess per image at shape (1, 3, 640, 416)
[[308, 412, 198, 207], [37, 410, 265, 208], [265, 53, 286, 160], [266, 226, 277, 179], [33, 629, 52, 207], [39, 30, 254, 320]]
Panel List after sorting top to bottom [[39, 30, 254, 320], [265, 53, 286, 160], [266, 226, 277, 179], [37, 410, 265, 208], [308, 412, 198, 207], [33, 629, 52, 207]]
[[39, 30, 254, 320], [265, 53, 286, 160], [266, 226, 277, 179], [37, 410, 265, 208], [308, 412, 198, 207], [33, 629, 52, 207]]
Cropping Images
[39, 30, 254, 320]
[265, 53, 286, 160]
[266, 226, 277, 179]
[37, 410, 265, 208]
[308, 412, 198, 207]
[33, 629, 52, 207]
Extracting Text


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

The image processor of type `Glm46VImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

voices/af_heart.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

0 I'm going to bed.I, what's gotten into her?  She just— ding dongOk I almost forgot! There's Mohammed pulling up his comics!  Can you answer that while I grab them from the studio?  Seriously?Hi You must be like Me Or- Died? I'm a friend of your dad?  I-Miles, yeah, How'd you know-Ok,uh you and your brother were mentioned in a 1989 Attouning Tales after view.  Plus! You guys were in the Donkin issue 56!--- ˌIm ɡˈOɪŋ tə bˈɛd.ˌI, wˌʌts ɡˈɑtn ˈɪntu hˌɜɹ? ʃi ʤˈʌst— dˈɪŋ dˌɔŋˈOkˌA ˌI ˈɔlmOst fəɹɡˈɑt! ðˌɛɹz mOhˈæmɪd pˈʊlɪŋ ˌʌp hɪz kˈɑmɪks! kˈæn ju ˈænsəɹ ðˈæt wˌIl ˌI ɡɹˈæb ðˌɛm fɹʌm ðə stˈudiO? sˈɪɹiəsli?hˈI jˌu mˈʌst bi lˈIk mˌi ˌɔɹ dˈId? ˌIm ɐ fɹˈɛnd ʌv jʊɹ dˈæd? ImˈIlz, jˈɛə, hˌWd ju nˌOˈOkˌA,ˈʌ ju ænd jʊɹ bɹˈʌðəɹ wɜɹ mˈɛnʧᵊnd ɪn ɐ nˌIntˈin ˈATi nˈIn ətˈWnɪŋ tˈAlz ˈæftəɹ vjˈu. plˈʌs! jˌu ɡˈIz wɜɹ ɪn ðə dˈɔŋkɪn ˈɪʃu fˈɪfti sˈɪks!
